In [2]:
from d2l import torch as d2l
import torch

In [2]:
def load_array(data_arrays, batch_size, is_train=True):
    """Construct a PyTorch data iterator.
    
    data_arrays: pytorch data, must have dimension 
    batch_size: the size of feature in each iter 
    is_train: shuffle or not
    """
    dataset = torch.utils.data.TensorDataset(*data_arrays)
    return torch.utils.data.DataLoader(dataset, batch_size, shuffle=is_train)

In [1]:
import torch
from torch import nn
from torchvision.transforms import ToPILImage
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader

def tensor_to_PILimage(net: nn.Module, 
                       train_iter: DataLoader | torch.Tensor,
                       tensor_index: int=0,
                       columns: int=8,
                       figsize: tuple=(2,2),
                      ) -> None:
    '''
    tensor_index: the index of the tensor to choose
    columns: the number of plots in one row
    figsize: tuple of (width,height)
    '''
    if isinstance(net,nn.DataParallel):
        net=net.module.cpu()
        
    if isinstance(train_iter,DataLoader):
        X=train_iter.dataset[tensor_index][0]
        #y=train_iter.dataset[tensor_index][1]
    elif isinstance(train_iter,torch.Tensor):
        X=train_iter
    else:
        raise "Wrong train_iter"
        
    plot_width,plot_height=figsize
    fig, axes = plt.subplots(1, 1, figsize=(2, 2))
    image_X=ToPILImage()(X)
    axes.imshow(image_X)
    
    layer=0
    y_hat=net[layer](X)
    to_pil=ToPILImage()
    while not isinstance(net[layer],nn.Flatten):
        chanel=y_hat.shape[0]
        #height=y_hat.shape[1]
        #width=y_hat.shape[2]
        row=int((chanel-0.9)//columns+1)  #否则恰好整除时会多 1 
        fig, axes = plt.subplots(row, columns,
                                 figsize=(plot_width*columns, plot_height*row))
        for i in range(chanel):
            image = to_pil(y_hat[[i]])
            if row==1:
                axes[i].imshow(image)
                axes[0].set(title=f'net{layer} '+type(net[layer]).__name__)
            else:
                axes[i//columns,i%columns].imshow(image) #实际上是灰度的,是matplotlib带了颜色
                axes[0,0].set(title=f'net{layer} '+type(net[layer]).__name__)
        layer+=1
        y_hat=net[layer](y_hat)


In [2]:
def save_checkpoint(epoch, net, optimizer, checkpoint_path):
    checkpoint_path=checkpoint_path if isinstance(checkpoint_path,str) else 'checkpoint.pth'
    # 待添加功能:输出路径 
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': net.module.state_dict() if isinstance(net,nn.DataParallel) \
                          else net.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }                        
    torch.save(checkpoint, checkpoint_path)
    return f'save checkpoint at epoch {epoch}'

In [11]:
def train_GPU_save(net, train_iter, test_iter, num_epochs, lr, device, weight_decay=0, 
              save_checkpoints: str =True, use_checkpoints: str =None,
              save_net: str =True,
             ):
    """
    Train a model with a GPU .
    You can save checkpoints and the net . It's stronger.

    """
    def init_weights(m):
        if type(m) == nn.Linear or type(m) == nn.Conv2d:
            nn.init.xavier_uniform_(m.weight)
    net.apply(init_weights)
    print('training on', device)

    optimizer = torch.optim.SGD(net.parameters(), lr=lr, weight_decay=weight_decay)
    loss = nn.CrossEntropyLoss()
    start_epoch=0 #用于使用checkpoint之后可以增量

    animator = d2l.Animator(xlabel='epoch', xlim=[1, num_epochs],
                            legend=['train loss', 'train acc', 'test acc'])
    timer, num_batches = d2l.Timer(), len(train_iter)

    best_test_acc = 0.0 #用于保存网络时选择最优的网络
    if save_net:
        # 确定模型保存路径：若 save_net 是字符串则用作路径，否则使用默认文件名
        best_model_path = save_net if isinstance(save_net, str) else 'best_model.pth'

    if use_checkpoints:
        try:
            checkpoint = torch.load(use_checkpoints, weights_only=True)
            if isinstance(net, nn.DataParallel):
                net.module.load_state_dict(checkpoint['model_state_dict'])
            else:
                net.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch=checkpoint['epoch']+1
            print('Use checkpoint successfully!')
        except FileNotFoundError:
            print('ATTENTION: There is not the checkpoint.')
        except Exception:
            raise
    
    net.to(device)
    status='No saved checkpoints' #防止没有调用save_checkpoint函数时没有status变量
    for epoch in range(start_epoch,num_epochs):
        # Sum of training loss, sum of training accuracy, no. of examples
        metric = d2l.Accumulator(3)
        net.train()
        for i, (X, y) in enumerate(train_iter):
            timer.start()
            optimizer.zero_grad()
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            l = loss(y_hat, y)
            l.backward()
            optimizer.step()
            with torch.no_grad():
                metric.add(l * X.shape[0], d2l.accuracy(y_hat, y), X.shape[0])
            timer.stop()
            train_l = metric[0] / metric[2]
            train_acc = metric[1] / metric[2]
            if (i + 1) % (num_batches // 5) == 0 or i == num_batches - 1:
                animator.add(epoch + (i + 1) / num_batches,
                             (train_l, train_acc, None))
        
        if epoch%round(num_epochs*0.25)==0 and save_checkpoints:
           status=save_checkpoint(epoch, net, optimizer, save_checkpoints)
        
        test_acc = d2l.evaluate_accuracy_gpu(net, test_iter)
        animator.add(epoch + 1, (None, None, test_acc))

        if save_net:
            if test_acc > best_test_acc:
                best_test_acc = test_acc
                # 处理 DataParallel 包装的模型
                model_to_save = net.module if isinstance(net, nn.DataParallel) else net
                torch.save(model_to_save.state_dict(), best_model_path)
                print(f'Best model saved with test acc {test_acc:.3f} to {best_model_path}')

        
    print(f'loss {train_l:.3f}, train acc {train_acc:.3f}, '
          f'test acc {test_acc:.3f}')
    print(f'{metric[2] * num_epochs / timer.sum():.1f} examples/sec \n'
          f'total time:{timer.sum():.4f} sec '
          f'on {str(device)}')
    print(status)
        

In [22]:
def jupyter_set_axes(axes, xlabel, ylabel, xlim, ylim, xscale, yscale, legend):
    """Set the axes for matplotlib.

    Defined in :numref:`sec_calculus`"""
    axes.set_xlabel(xlabel), axes.set_ylabel(ylabel)
    axes.set_xscale(xscale), axes.set_yscale(yscale)
    axes.set_xlim(xlim),     axes.set_ylim(ylim)
    if legend:
        axes.legend(legend)
    axes.grid()
    
class JupyterAnimator:
    import matplotlib.pyplot as plt
    from IPython import display
    """For plotting data in animation."""
    def __init__(self, xlabel=None, ylabel=None, legend=None, xlim=None,
                 ylim=None, xscale='linear', yscale='linear',
                 fmts=('-', 'm--', 'g-.', 'r:'), nrows=1, ncols=1,
                 figsize=(3.5, 2.5)):
        """
        """
        # Incrementally plot multiple lines

        
        if legend is None:
            legend = []
        from matplotlib_inline.backend_inline import set_matplotlib_formats
        set_matplotlib_formats('svg')
        
        self.fig, self.axes = plt.subplots(nrows, ncols, figsize=figsize)
               
        if nrows * ncols == 1:
            self.axes = [self.axes, ]
        # Use a lambda function to capture arguments
        self.config_axes = lambda:jupyter_set_axes(self.axes[0], xlabel, ylabel, 
                                           xlim, ylim, xscale, yscale, legend)
        self.X, self.Y, self.fmts = None, None, fmts

    def add(self, x, y):
        # Add multiple data points into the figure
        if not hasattr(y, "__len__"):
            y = [y]
        n = len(y)
        if not hasattr(x, "__len__"):
            x = [x] * n
        if not self.X:
            self.X = [[] for _ in range(n)]
        if not self.Y:
            self.Y = [[] for _ in range(n)]
        for i, (a, b) in enumerate(zip(x, y)):
            if a is not None and b is not None:
                self.X[i].append(a)
                self.Y[i].append(b)
        self.axes[0].cla()
        for x, y, fmt in zip(self.X, self.Y, self.fmts):
            self.axes[0].plot(x, y, fmt)
        self.config_axes()
        display.display(self.fig)
        display.clear_output(wait=True)

In [2]:
def draw_parameter_BN(net):
    '''
    draw gamma and beta of the BatchNorm layer
    also can be used for others
    '''
    import matplotlib.pyplot as plt
    %matplotlib inline
    import math
    count=0
    weight=[]
    bias=[]
    for layer in net:
        if isinstance(layer,(nn.BatchNorm1d, nn.BatchNorm2d)):
            count+=1
            layer_dict=dict(layer.named_parameters())
            weight.append(layer_dict['weight'].data)
            bias.append(layer_dict['bias'].data)
    fig,ax=plt.subplots(math.ceil(count/2),2,figsize=(6*2,4*math.ceil(count/2)),sharey=True,)
    for i in range(count):
        weight_i=weight[i]
        bias_i=bias[i]
        x=torch.arange(weight_i.shape[0])
        ax[i//2,i%2].scatter(x=x,y=weight_i,label='weight')
        ax[i//2,i%2].plot(weight_i.mean().expand(weight_i.shape[0]),color='pink',label='mean_weight')
        ax[i//2,i%2].scatter(x=x,y=bias_i,color='purple',label='bias')
        ax[i//2,i%2].plot(bias_i.mean().expand(bias_i.shape[0]),color='red',linestyle='dotted',label='mean_bias')
        ax[i//2,i%2].text(0,5, f"std of weight {weight_i.std():.2f}",family="monospace", fontsize=10)
        ax[i//2,i%2].text(0,4.5, f"std of bias {bias_i.std():.2f}",family="monospace", fontsize=10)
    
        ax[i//2,i%2].legend()

In [27]:
import matplotlib
matplotlib.use('Agg')          # 无头环境强制使用 Agg 后端
import matplotlib.pyplot as plt

def set_axes(axes, xlabel, ylabel, xlim, ylim, xscale, yscale, legend):
    """设置坐标轴属性（与原始一致）"""
    axes.set_xlabel(xlabel)
    axes.set_ylabel(ylabel)
    axes.set_xscale(xscale)
    axes.set_yscale(yscale)
    if xlim is not None:
        axes.set_xlim(xlim)
    if ylim is not None:
        axes.set_ylim(ylim)
    if legend:
        axes.legend(legend)
    axes.grid()

class AnimatorServer:
    """服务器端动画记录器：收集数据，支持随时保存为图片"""
    def __init__(self, xlabel=None, ylabel=None, legend=None, xlim=None,
                 ylim=None, xscale='linear', yscale='linear',
                 fmts=('-', 'm--', 'g-.', 'r:'), nrows=1, ncols=1,
                 figsize=(3.5, 2.5)):
        if legend is None:
            legend = []
        
        self.fig, self.axes = plt.subplots(nrows, ncols, figsize=figsize)
        if nrows * ncols == 1:
            self.axes = [self.axes]
        
        # 配置坐标轴的函数（只用于初始化）
        self.config_axes = lambda: set_axes(
            self.axes[0], xlabel, ylabel, xlim, ylim, xscale, yscale, legend)
        
        self.fmts = fmts
        self.X, self.Y = None, None
        self.lines = []          # 预创建的线条对象
        self._init_lines()

    def _init_lines(self):
        """预先创建空线条，后续只更新数据"""
        for fmt in self.fmts:
            line, = self.axes[0].plot([], [], fmt)
            self.lines.append(line)
        self.config_axes()       # 固定轴样式

    def add(self, x, y):
        """添加数据点并更新图形（不显示，不保存）"""
        # 格式化输入
        if not hasattr(y, "__len__"):
            y = [y]
        n = len(y)
        if not hasattr(x, "__len__"):
            x = [x] * n
        if self.X is None:
            self.X = [[] for _ in range(n)]
        if self.Y is None:
            self.Y = [[] for _ in range(n)]
        
        for i, (a, b) in enumerate(zip(x, y)):
            if a is not None and b is not None:
                self.X[i].append(a)
                self.Y[i].append(b)
        
        # 动态扩展线条数量（若数据曲线多于预设样式）
        while len(self.lines) < len(self.X):
            fmt = self.fmts[len(self.lines) % len(self.fmts)]
            line, = self.axes[0].plot([], [], fmt)
            self.lines.append(line)
        
        # 更新所有线条数据
        for line, xdata, ydata in zip(self.lines, self.X, self.Y):
            line.set_data(xdata, ydata)
        
        # 自动调整坐标轴范围
        ax = self.axes[0]
        ax.relim()
        ax.autoscale_view()
        # 处理单点情况，留出适当边距
        if len(self.X[0]) == 1:
            ax.set_xlim(self.X[0][0] - 0.5, self.X[0][0] + 0.5)
            ax.set_ylim(self.Y[0][0] - 0.5, self.Y[0][0] + 0.5)

    def save(self, path):
        """将当前图形保存为文件（支持 .png, .pdf, .svg 等）"""
        self.fig.savefig(path, dpi=100, bbox_inches='tight')
        print(f"Plot saved to {path}")

    def close(self):
        """释放图形资源"""
        plt.close(self.fig)

# 还需要测试

In [28]:
# 创建记录器
anim = AnimatorServer(xlabel='epoch', ylabel='loss', legend=['train', 'val'],
                      xlim=[0, 10], ylim=[0, 5])

# 模拟训练过程，随时保存快照
for epoch in range(10):
    train_loss = 3.0 / (epoch + 1)
    val_loss = 2.5 / (epoch + 1) + 0.1
    anim.add(epoch, [train_loss, val_loss])
    
    # 第 5 个 epoch 时保存一张图
    if epoch == 3:
        anim.save('loss_epoch3.png')

# 训练结束后保存最终曲线
anim.save('loss_final.png')
anim.close()

Plot saved to loss_epoch3.png
Plot saved to loss_final.png


In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from typing import Optional

def analyze_parameter_tensor(
    tensor: torch.Tensor,
    name: Optional[str] = None,
    plot_hist: bool = True,
    bins: int = 50,
    figsize: tuple = (10, 4),
    show_stats: bool = True,
    show_values: bool = False,
    num_values: int = 10,
) -> dict:
    """
    分析神经网络参数张量的统计特性（适用于权重、偏置等）。
    
    Args:
        tensor: 待分析的 torch.Tensor
        name: 参数名称（如 'layer1.weight'），用于打印和标题
        plot_hist: 是否绘制直方图
        bins: 直方图的 bin 数量
        figsize: 图像大小（宽, 高）
        show_stats: 是否打印统计摘要
        show_values: 是否显示前几个元素的值
        num_values: 若 show_values=True，显示的元素个数
        
    Returns:
        dict: 包含统计信息的字典
    """
    # 处理设备、梯度
    if tensor.is_cuda:
        tensor = tensor.cpu()
    if tensor.requires_grad:
        tensor = tensor.detach()
    
    # 展平为一维数组以便统计
    flat = tensor.flatten().numpy()
    
    # 计算统计量
    min_val = flat.min()
    max_val = flat.max()
    mean_val = flat.mean()
    std_val = flat.std()
    median_val = np.median(flat)
    q25 = np.percentile(flat, 25)
    q75 = np.percentile(flat, 75)
    iqr = q75 - q25
    skewness = stats.skew(flat)
    kurtosis = stats.kurtosis(flat)
    # 非零比例（稀疏度）
    nonzero_ratio = np.count_nonzero(flat) / flat.size
    # 绝对值均值（L1 范数均值）
    abs_mean = np.abs(flat).mean()
    
    stats_dict = {
        'shape': tensor.shape,
        'numel': flat.size,
        'min': min_val,
        'max': max_val,
        'mean': mean_val,
        'std': std_val,
        'median': median_val,
        'q25': q25,
        'q75': q75,
        'iqr': iqr,
        'skewness': skewness,
        'kurtosis': kurtosis,
        'nonzero_ratio': nonzero_ratio,
        'abs_mean': abs_mean,
    }
    
    if show_stats:
        print("=" * 60)
        if name:
            print(f"Parameter: {name}")
        print(f"Shape: {tensor.shape}  |  Total elements: {flat.size:,}")
        print(f"Min: {min_val:.6f}  |  Max: {max_val:.6f}")
        print(f"Mean: {mean_val:.6f}  |  Std: {std_val:.6f}")
        print(f"Median: {median_val:.6f}  |  IQR: {iqr:.6f}")
        print(f"Skewness(偏度): {skewness:.4f}  |  Kurtosis(峰度): {kurtosis:.4f}")
        print(f"Non-zero ratio: {nonzero_ratio:.4%}  |  |value| mean: {abs_mean:.6f}")
        print("=" * 60)
    
    if show_values:
        # 显示前 num_values 个元素（展平后的）
        sample_vals = flat[:num_values]
        print(f"First {min(num_values, len(sample_vals))} values: {sample_vals}")
    
    if plot_hist:
        plt.figure(figsize=figsize)
        # 根据数据分布特性选择合适的颜色和透明度
        plt.hist(flat, bins=bins, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.5)
        # 添加均值、中位数线
        plt.axvline(mean_val, color='red', linestyle='dashed', linewidth=1.5, label=f'Mean = {mean_val:.4f}')
        plt.axvline(median_val, color='green', linestyle='dashed', linewidth=1.5, label=f'Median = {median_val:.4f}')
        plt.xlabel('Value')
        plt.ylabel('Frequency')
        title = f"Histogram of {name}" if name else "Parameter value distribution"
        plt.title(title)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    return stats_dict